# Local LLM-Based Development Intelligence System  
## Montenegro Human Development Report 2009

This notebook builds a complete PDF-to-dashboard pipeline using the Montenegro National Human Development Report 2009. The system extracts text from the PDF, cleans and chunks the document, uses local LLMs for summarisation and structured extraction, evaluates the outputs using a second LLM, and visualises the extracted development indicators in a dashboard-style format.

In [1]:
pip install pymupdf pandas numpy matplotlib plotly streamlit ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
import fitz
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ollama
from pathlib import Path

## 1. PDF Processing Pipeline

The first stage loads the UN Human Development Report PDF, extracts raw text, cleans unnecessary formatting, and splits the report into smaller text chunks. This prepares the document for local LLM processing.

In [3]:
PDF_PATH = "montenegronhdr2009en.pdf"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

EXTRACTION_MODEL = "llama3"
EVALUATION_MODEL = "mistral:7b"

In [4]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        pages.append({
            "page": page_num,
            "text": text
        })

    return pages


pages = extract_text_from_pdf(PDF_PATH)
print(f"Extracted text from {len(pages)} pages")

Extracted text from 122 pages


In [5]:
def clean_text(text):
    text = text.replace("￾", "")
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


cleaned_pages = []

for page in pages:
    cleaned_pages.append({
        "page": page["page"],
        "text": clean_text(page["text"])
    })

full_text = "\n\n".join([p["text"] for p in cleaned_pages])

with open(OUTPUT_DIR / "montenegro_cleaned_text.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("Cleaned text saved.")
print(full_text[:1000])

Cleaned text saved.
National Human Development Report 2009 Montenegro: Society for All Montenegro

1 National Human Development Report 2009 Montenegro: Society for All National Human Development Report 2009 Montenegro: Society for All

2 National Human Development Report 2009 Montenegro: Society for All United Nations Development Programme (UNDP) is the UN’s global development network, advocating for change and connecting countries to knowledge, experience and resources to help people build a better life. We are on the ground in 166 countries, working with them on their own solutions to global and national development challenges. As they develop local capacity, they draw on the people of UNDP and our wide range of partners. Short extracts from this publication may be reproduced unaltered without authorisation, on condition that the source is indicated. The views expressed in this paper are those of the authors and do not necessarily represent the views of UNDP. Copyright © 2009 By the 

In [6]:
def create_chunks(text, chunk_size=600, overlap=150):
    words = text.split()
    chunks = []
    start = 0
    chunk_id = 1

    while start < len(words):
        end = start + chunk_size
        chunk_text = " ".join(words[start:end])

        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk_text
        })

        chunk_id += 1
        start += chunk_size - overlap

    return chunks


chunks = create_chunks(full_text)

with open(OUTPUT_DIR / "text_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Created {len(chunks)} chunks")

Created 142 chunks


## 2. Local LLM Summarisation

One local LLM is used to summarise the full report and individual chapters. The report focuses on poverty, human development, social exclusion, vulnerable groups, regional disparities, and policy recommendations.

In [7]:
def ask_ollama(model, prompt):
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response["message"]["content"]
    

In [8]:
summary_prompt = f"""
You are analysing the Montenegro National Human Development Report 2009.

Task:
Provide 6 concise bullet points summarising the key findings of the report.

Focus on:
- human development
- poverty
- inequality
- vulnerable groups
- social exclusion
- regional disparities
- policy recommendations

Use only the source text below.

SOURCE TEXT:
{full_text[:5000]}
"""

key_findings = ask_ollama(EXTRACTION_MODEL, summary_prompt)

print(key_findings)

with open(OUTPUT_DIR / "key_findings.txt", "w", encoding="utf-8") as f:
    f.write(key_findings)

Based on the Montenegro National Human Development Report 2009, here are six concise bullet points summarizing the key findings:

• **Human development**: The report highlights that Montenegro has made significant progress in human development, with improvements in education, health, and living standards. However, there is still a gap between the rich and the poor, and vulnerable groups continue to face challenges.

• **Poverty**: Poverty remains a significant issue in Montenegro, affecting approximately 12% of the population. The report notes that poverty is more prevalent among certain groups, including those living in rural areas, ethnic minorities, and individuals with disabilities.

• **Inequality**: The report emphasizes that inequality is a major challenge in Montenegro, with significant disparities in wealth, income, and access to opportunities. This is particularly evident in the difference between urban and rural areas, where poverty rates are higher in rural regions.

• **Vu

In [9]:
chapter_texts = {
    "Chapter 1": " ".join([p["text"] for p in cleaned_pages[11:17]]),
    "Chapter 2": " ".join([p["text"] for p in cleaned_pages[17:35]]),
    "Chapter 3": " ".join([p["text"] for p in cleaned_pages[35:77]]),
    "Chapter 4": " ".join([p["text"] for p in cleaned_pages[77:85]]),
    "Chapter 5": " ".join([p["text"] for p in cleaned_pages[85:103]])
}

for chapter, text in chapter_texts.items():
    print(chapter, len(text))
    print(text[:200])
    print("-" * 80)

Chapter 1 25455
11 National Human Development Report 2009 Montenegro: Society for All Chapter 1: Human development and social exclusion 12 National Human Development Report 2009 Montenegro: Society for All UNDP intro
--------------------------------------------------------------------------------
Chapter 2 67296
17 National Human Development Report 2009 Montenegro: Society for All Chapter 2: Status of Poverty, Human Development and Social Exclusion in Montenegro 18 National Human Development Report 2009 Monte
--------------------------------------------------------------------------------
Chapter 3 167549
35 National Human Development Report 2009 Montenegro: Society for All Chapter 3: Vulnerable groups 36 National Human Development Report 2009 Montenegro: Society for All Certain groups in the Montenegr
--------------------------------------------------------------------------------
Chapter 4 25854
77 National Human Development Report 2009 Montenegro: Society for All Chapter 4. Regional

In [10]:
chapter_summaries = {}

for chapter, text in chapter_texts.items():
    prompt = f"""
    Summarise {chapter} of the Montenegro Human Development Report in fewer than 100 words.

    Use only the text below.

    TEXT:
    {text[:4000]}
    """

    summary = ask_ollama(EXTRACTION_MODEL, prompt)
    chapter_summaries[chapter] = summary

chapter_summaries_df = pd.DataFrame(
    list(chapter_summaries.items()),
    columns=["chapter", "summary"]
)

chapter_summaries_df.to_csv(OUTPUT_DIR / "chapter_summaries.csv", index=False)

chapter_summaries_df

,chapter,summary
0,Chapter 1,Here is a summary of Chapter 1 of the Monteneg...
1,Chapter 2,Here is a summary of Chapter 2 of the Monteneg...
2,Chapter 3,Here is a summary of Chapter 3 of the Monteneg...
3,Chapter 4,Here is a summary of Chapter 4 of the Monteneg...
4,Chapter 5,Here is a summary of Chapter 5 of the Monteneg...


## 3. Thematic Extraction

The next stage counts how often key development themes appear in the report. The selected themes are education, health, inequality, economy, gender, climate, and employment.

In [11]:
themes = {
    "education": ["education", "school", "literacy", "enrolment", "university"],
    "health": ["health", "healthcare", "doctor", "mortality", "life expectancy"],
    "inequality": ["inequality", "poverty", "exclusion", "vulnerable", "discrimination"],
    "economy": ["economy", "economic", "GDP", "income", "budget", "growth"],
    "gender": ["gender", "women", "female", "girls"],
    "climate": ["climate", "environment", "sustainability", "natural"],
    "employment": ["employment", "unemployment", "labour", "job", "work"]
}

theme_counts = {}

lower_text = full_text.lower()

for theme, keywords in themes.items():
    count = 0
    for keyword in keywords:
        count += lower_text.count(keyword.lower())
    theme_counts[theme] = count

theme_df = pd.DataFrame(
    list(theme_counts.items()),
    columns=["theme", "count"]
)

theme_df.to_csv(OUTPUT_DIR / "theme_counts.csv", index=False)

theme_df

,theme,count
0,education,398
1,health,351
2,inequality,727
3,economy,449
4,gender,145
5,climate,24
6,employment,739


In [12]:
strength_challenge_prompt = f"""
Extract development strengths and challenges from the Montenegro Human Development Report.

Return valid JSON only in this format:

{{
  "strengths": [
    "strength 1",
    "strength 2"
  ],
  "challenges": [
    "challenge 1",
    "challenge 2"
  ]
}}

Rules:
- Maximum 8 strengths
- Maximum 8 challenges
- Use concise wording
- Use only the source text
- Do not add explanations outside JSON

SOURCE TEXT:
{full_text[:15000]}
"""

strengths_challenges = ask_ollama(EXTRACTION_MODEL, strength_challenge_prompt)

print(strengths_challenges)

with open(OUTPUT_DIR / "strengths_challenges.json", "w", encoding="utf-8") as f:
    f.write(strengths_challenges)

Here is the extracted information in the requested format:

{
"strengths": [
"Economic growth during 2006-2007 was exceptional",
"Montenegro has enormous potential",
"Dramatic economic decline resulted in severe unemployment and hardships, but also created important new opportunities",
"The country's prospective EU membership will bring further opportunities for the citizens of Montenegro",
"Montenegro secured membership in international institutions",
"Verified a Stabilization and Association Agreement with the European Union (EU) in March 2007"
],
"challenges": [
"Global economic downturn has begun to take a toll on the people of Montenegro and especially the vulnerable and socially excluded",
"Economic and political challenges of the 1990s resulted in severe unemployment and dramatic economic decline",
"International sanctions created substantial hardships",
"Social exclusion persists, with 11% of the population continuing to live below the poverty line",
"Many individuals are or ri

In [24]:
strengths_challenges_clean = {
    "strengths": [
        "High adult literacy rate",
        "High gross enrolment rate",
        "Improving HDI after 1999",
        "Progress in EU integration",
        "Strong policy focus on social inclusion",
        "Available social sector data for vulnerable groups"
    ],
    "challenges": [
        "Poverty remains present among 11% of the population",
        "Regional disparities in development",
        "Long-term unemployment",
        "Social exclusion of vulnerable groups",
        "Limited access to services for some groups",
        "Lower development outcomes in northern regions"
    ]
}

with open(OUTPUT_DIR / "strengths_challenges.json", "w", encoding="utf-8") as f:
    json.dump(strengths_challenges_clean, f, indent=2)

strengths_challenges_clean

{'strengths': ['High adult literacy rate',
  'High gross enrolment rate',
  'Improving HDI after 1999',
  'Progress in EU integration',
  'Strong policy focus on social inclusion',
  'Available social sector data for vulnerable groups'],
 'challenges': ['Poverty remains present among 11% of the population',
  'Regional disparities in development',
  'Long-term unemployment',
  'Social exclusion of vulnerable groups',
  'Limited access to services for some groups',
  'Lower development outcomes in northern regions']}

## 4. Structured Numerical Indicator Extraction

The system extracts core development indicators into JSON format. These indicators are used later for visual analytics and dashboard creation.

In [13]:
indicator_prompt = f"""
Extract numerical development indicators from the Montenegro Human Development Report.

Return valid JSON only using this structure:

{{
  "country": "Montenegro",
  "report_year": 2009,
  "indicators": {{
    "population": null,
    "life_expectancy": null,
    "adult_literacy_rate": null,
    "gross_enrolment_rate": null,
    "GDP_per_capita_PPP": null,
    "HDI_rank": null,
    "HDI_value": null,
    "poverty_rate": null,
    "infant_mortality_rate": null
  }}
}}

Rules:
- Use numbers where possible
- Use null if the value is not found
- Do not invent values
- Return JSON only

SOURCE TEXT:
{full_text[:15000]}
"""

indicator_json = ask_ollama(EXTRACTION_MODEL, indicator_prompt)

print(indicator_json)

with open(OUTPUT_DIR / "extracted_indicators.json", "w", encoding="utf-8") as f:
    f.write(indicator_json)

Here is the extracted numerical development indicators in JSON format:

{
  "country": "Montenegro",
  "report_year": 2009,
  "indicators": {
    "population": 625000,
    "life_expectancy": 72.7,
    "adult_literacy_rate": 97.7,
    "gross_enrolment_rate": 80.7,
    "GDP_per_capita_PPP": 9934.6,
    "HDI_rank": 64/179,
    "HDI_value": null,
    "poverty_rate": 11,
    "infant_mortality_rate": 11
  }
}

Note that some indicators, such as HDI value and life expectancy for males and females, are not explicitly mentioned in the provided text. I have filled those with `null` values as per your instruction.


In [23]:
manual_indicators = {
    "country": "Montenegro",
    "report_year": 2009,
    "population": 625000,
    "life_expectancy": 72.7,
    "adult_literacy_rate": 97.7,
    "gross_enrolment_rate": 80.7,
    "GDP_per_capita_PPP": 9934.6,
    "HDI_rank": 64,
    "HDI_total_countries": 179,
    "HDI_value": 0.828,
    "poverty_rate": 11,
    "infant_mortality_rate": 11
}

with open(OUTPUT_DIR / "clean_indicators.json", "w", encoding="utf-8") as f:
    json.dump(manual_indicators, f, indent=2)

indicator_df = pd.DataFrame(
    list(manual_indicators.items()),
    columns=["indicator", "value"]
)

indicator_df.to_csv(OUTPUT_DIR / "manual_indicators.csv", index=False)

indicator_df

,indicator,value
0,country,Montenegro
1,report_year,2009
2,population,625000
3,life_expectancy,72.7
4,adult_literacy_rate,97.7
5,gross_enrolment_rate,80.7
6,GDP_per_capita_PPP,9934.6
7,HDI_rank,64
8,HDI_total_countries,179
9,HDI_value,0.828


## 5. Demographic and HDI Trend Extraction

The report contains time-based indicators that can be plotted, especially HDI values over time and life expectancy changes.

In [15]:
hdi_trend = pd.DataFrame({
    "year": [1991, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007],
    "HDI": [0.789, 0.755, 0.775, 0.771, 0.775, 0.797, 0.804, 0.805, 0.816, 0.828]
})

life_expectancy_trend = pd.DataFrame({
    "year": [1991, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007],
    "life_expectancy": [75.2, 73.4, 73.4, 73.4, 73.0, 73.1, 73.1, 72.6, 72.7, 72.7]
})

hdi_trend.to_csv(OUTPUT_DIR / "hdi_trend.csv", index=False)
life_expectancy_trend.to_csv(OUTPUT_DIR / "life_expectancy_trend.csv", index=False)

hdi_trend

,year,HDI
0,1991,0.789
1,1999,0.755
2,2000,0.775
3,2001,0.771
4,2002,0.775
5,2003,0.797
6,2004,0.804
7,2005,0.805
8,2006,0.816
9,2007,0.828


## 6. LLM Evaluation Framework

A second local LLM is used as an evaluator. It assesses whether the extracted outputs are complete, consistent, and factually aligned with the source report.

In [16]:
evaluation_prompt = f"""
You are an evaluator LLM.

Evaluate the quality of the following extracted output from the Montenegro Human Development Report.

Assess using these criteria:
1. Completeness
2. Consistency
3. Factual alignment with source text
4. Usefulness for dashboard visualisation

Give a score from 1 to 5 for each criterion and provide short comments.

SOURCE TEXT:
{full_text[:10000]}

EXTRACTED OUTPUT:
{indicator_json}
"""

evaluation_result = ask_ollama(EVALUATION_MODEL, evaluation_prompt)

print(evaluation_result)

with open(OUTPUT_DIR / "llm_evaluation.txt", "w", encoding="utf-8") as f:
    f.write(evaluation_result)

 Criteria | Score | Comments
---------|-------|---------------
Completeness | 4 | The extracted output is largely complete but misses some details such as the specific years for certain indicators (e.g., life expectancy, infant mortality rate) and does not include all the indicators mentioned in the source text (e.g., number of doctors per 100,000 inhabitants, access to information, etc.).
Consistency | 4 | The extracted output is consistent with the source text in terms of the reported numerical indicators for Montenegro's development, but the formatting and organization are different.
Factual alignment with source text | 5 | The factual data presented in the extracted output is accurate and aligned with the data provided in the source text.
Usefulness for dashboard visualisation | 4 | The extracted output provides a useful summary of numerical development indicators for Montenegro that could be easily represented in a dashboard format, although it may require additional formatting or

## 7. Data Visualisation

The dashboard includes at least four plots:
1. Theme distribution  
2. Core development indicators  
3. HDI trend over time  
4. Life expectancy trend over time  
5. Advanced radar chart of development indicators  

In [33]:
pip install nbformat ipykernel

Note: you may need to restart the kernel to use updated packages.


In [17]:
fig1 = px.bar(
    theme_df,
    x="theme",
    y="count",
    title="Distribution of Development Themes in the Report",
    labels={"theme": "Theme", "count": "Keyword Count"}
)

fig1.show()
fig1.write_html(OUTPUT_DIR / "theme_distribution.html")

In [18]:
selected_indicators = pd.DataFrame({
    "indicator": [
        "Life Expectancy",
        "Adult Literacy Rate",
        "Gross Enrolment Rate",
        "Poverty Rate",
        "Infant Mortality Rate"
    ],
    "value": [
        72.7,
        97.7,
        80.7,
        11,
        11
    ]
})

fig2 = px.bar(
    selected_indicators,
    x="indicator",
    y="value",
    title="Selected Development Indicators for Montenegro",
    labels={"indicator": "Indicator", "value": "Value"}
)

fig2.show()
fig2.write_html(OUTPUT_DIR / "core_indicators.html")

In [19]:
fig3 = px.line(
    hdi_trend,
    x="year",
    y="HDI",
    markers=True,
    title="Human Development Index Trend in Montenegro"
)

fig3.show()
fig3.write_html(OUTPUT_DIR / "hdi_trend.html")

In [20]:
fig4 = px.line(
    life_expectancy_trend,
    x="year",
    y="life_expectancy",
    markers=True,
    title="Life Expectancy Trend in Montenegro"
)

fig4.show()
fig4.write_html(OUTPUT_DIR / "life_expectancy_trend.html")

In [21]:
radar_df = pd.DataFrame({
    "indicator": [
        "Life Expectancy",
        "Adult Literacy",
        "Gross Enrolment",
        "Poverty Reduction",
        "HDI Score"
    ],
    "value": [
        72.7,
        97.7,
        80.7,
        89,
        82.8
    ]
})

fig5 = go.Figure()

fig5.add_trace(go.Scatterpolar(
    r=radar_df["value"],
    theta=radar_df["indicator"],
    fill="toself",
    name="Montenegro"
))

fig5.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    title="Radar Chart of Montenegro Development Indicators"
)

fig5.show()
fig5.write_html(OUTPUT_DIR / "radar_chart.html")

## 8. Dashboard Summary

The visual outputs show that the report focuses strongly on social exclusion, poverty, employment, health, education, and inequality. The HDI trend shows improvement after 1999, while the report also highlights continuing issues such as poverty, vulnerable groups, regional disparities, and unequal access to services.

In [22]:
dashboard_data = {
    "theme_counts": theme_counts,
    "manual_indicators": manual_indicators,
    "chapter_summaries": chapter_summaries
}

with open(OUTPUT_DIR / "dashboard_data.json", "w", encoding="utf-8") as f:
    json.dump(dashboard_data, f, indent=2, ensure_ascii=False)

print("All dashboard data saved successfully.")

All dashboard data saved successfully.


## Conclusion

This notebook demonstrates a complete local LLM pipeline for development intelligence extraction from a UN Human Development Report. The system extracts and cleans PDF text, chunks the document, summarises chapters, extracts structured indicators, evaluates LLM output quality, and produces dashboard-ready visualisations. The results show that Montenegro had a high level of human development by 2007, but still faced important challenges around poverty, unemployment, social exclusion, vulnerable groups, and regional inequality.